# Импорты, пути, данные

## Импорты

In [2]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl

import lightgbm as lgb
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_auc_score

import joblib

import os

import numpy as np
from pathlib import Path

## Пути

In [3]:
path_train = "data\\train\\"
path_test = "data\\test\\"
path_sample_submit = "data\\sample_submit.parquet"
path_summit = "data\\submission\\"

path_save_model_all_feature = "data\model\model_baseline_all_feature\\"
path_save_model_part_feature = "data\model\model_baseline_part_feature\\"
path_save_model_minimum_feature = "data\model\model_minimum_feature\\"
path_save_model_with_extra_feature = "data\model\model_extra_feature\\"
path_save_model_with_extra_feature2 = "data\model\model_extra_feature2\\"

path_m_train = f"{path_train}train_main_features.parquet"
path_e_train = f"{path_train}train_extra_features.parquet"
path_save_train = "data\\res_data\\train.parquet"

path_m_test = f"{path_test}test_main_features.parquet"
path_e_test = f"{path_test}test_extra_features.parquet"
path_save_test = "data\\res_data\\test.parquet"

## Загрузка и обработка данных

### Основа

In [4]:
def _read_feature_list(path):
    with open(path, "r", encoding="utf-8") as file:
        return [line.strip().strip("'").strip("',") for line in file if line.strip()]


def _safe_float32(df, cols):
    for col in cols:
        if col in df.columns and df[col].dtype == "float64":
            df[col] = df[col].astype("float32")


def _concat_features(df, new_cols):
    if not new_cols:
        return df
    new_df = pd.DataFrame(new_cols, index=df.index)
    return pd.concat([df, new_df], axis=1)


def _add_pairwise_features(df, feature_list):
    valid_features = [col for col in feature_list if col in df.columns]
    new_cols = {}

    for i, col1 in enumerate(valid_features):
        s1 = df[col1]
        for col2 in valid_features[i + 1:]:
            s2 = df[col2]

            new_cols[f"{col1}_x_{col2}"] = (s1 * s2).astype("float32")
            new_cols[f"{col1}_sub_{col2}"] = (s1 - s2).astype("float32")
            new_cols[f"{col1}_div_{col2}"] = (s1 / s2.replace(0, np.nan)).astype("float32")
            new_cols[f"{col1}_gt_{col2}"] = (s1 > s2).astype("int8")

    return _concat_features(df, new_cols)


def _add_top100_base_features_train(df, top100_feature):
    freq_maps = {}
    quantile_bins = {}
    valid_features = [col for col in top100_feature if col in df.columns]
    new_cols = {}

    for col in valid_features:
        s = df[col]

        freq = s.value_counts(dropna=False)
        freq_maps[col] = freq
        new_cols[f"{col}_freq"] = s.map(freq).astype("float32")

        new_cols[f"{col}_isna"] = s.isna().astype("int8")
        new_cols[f"{col}_abs"] = s.abs().astype("float32")
        new_cols[f"{col}_rank_pct"] = s.rank(pct=True).astype("float32")

        try:
            qcut_res, bins = pd.qcut(
                s,
                q=10,
                labels=False,
                duplicates="drop",
                retbins=True,
            )
            new_cols[f"{col}_qbin"] = qcut_res.astype("float32")
            quantile_bins[col] = bins
        except ValueError:
            new_cols[f"{col}_qbin"] = pd.Series(np.nan, index=df.index, dtype="float32")
            quantile_bins[col] = None

    df = _concat_features(df, new_cols)
    return df, freq_maps, quantile_bins


def _add_top100_base_features_test(df, top100_feature, freq_maps, quantile_bins):
    valid_features = [col for col in top100_feature if col in df.columns]
    new_cols = {}

    for col in valid_features:
        s = df[col]

        new_cols[f"{col}_freq"] = s.map(freq_maps[col]).astype("float32")
        new_cols[f"{col}_isna"] = s.isna().astype("int8")
        new_cols[f"{col}_abs"] = s.abs().astype("float32")
        new_cols[f"{col}_rank_pct"] = s.rank(pct=True).astype("float32")

        bins = quantile_bins[col]
        if bins is not None and len(bins) > 1:
            bins = np.unique(bins)
            if len(bins) > 1:
                new_cols[f"{col}_qbin"] = pd.cut(
                    s,
                    bins=bins,
                    labels=False,
                    include_lowest=True,
                ).astype("float32")
            else:
                new_cols[f"{col}_qbin"] = pd.Series(np.nan, index=df.index, dtype="float32")
        else:
            new_cols[f"{col}_qbin"] = pd.Series(np.nan, index=df.index, dtype="float32")

    return _concat_features(df, new_cols)


def _add_rowwise_aggregations(df, top100_feature):
    valid_features = [col for col in top100_feature if col in df.columns]
    if not valid_features:
        return df

    block = df[valid_features]
    new_cols = {
        "top100_mean": block.mean(axis=1).astype("float32"),
        "top100_std": block.std(axis=1).astype("float32"),
        "top100_nan_count": block.isna().sum(axis=1).astype("int16"),
    }
    return _concat_features(df, new_cols)


def _build_category_agg_maps(train, cat_cols, top30_feature, max_cat_cols=5):
    agg_maps = {}

    valid_cat_cols = [col for col in cat_cols if col in train.columns][:max_cat_cols]
    valid_num_cols = [col for col in top30_feature if col in train.columns]

    for cat_col in valid_cat_cols:
        agg_maps[cat_col] = {}

        for num_col in valid_num_cols:
            grp = train.groupby(cat_col, observed=True)[num_col]
            mean_map = grp.mean()

            agg_maps[cat_col][num_col] = {
                "mean": mean_map,
            }

    return agg_maps


def _apply_category_aggs(df, agg_maps):
    new_cols = {}

    for cat_col, num_dict in agg_maps.items():
        if cat_col not in df.columns:
            continue

        cat_series = df[cat_col]

        for num_col, stat_dict in num_dict.items():
            if num_col not in df.columns:
                continue

            mean_map = stat_dict["mean"]
            mean_col = f"{num_col}_by_{cat_col}_mean"

            mapped_mean = cat_series.map(mean_map).astype("float32")
            new_cols[mean_col] = mapped_mean
            new_cols[f"{num_col}_minus_{cat_col}_mean"] = (
                df[num_col] - mapped_mean
            ).astype("float32")

    return _concat_features(df, new_cols)


def processing_train_test(
    path_m_train,
    path_e_train,
    path_m_test,
    path_e_test,
    path_save_train="",
    path_save_test="",
):
    # ============
    # 1. TRAIN
    # ============
    df_m_train = pd.read_parquet(path_m_train)
    df_e_train = pd.read_parquet(path_e_train)
    train = df_m_train.merge(df_e_train, on="customer_id", how="left")
    del df_m_train, df_e_train

    drop_features = _read_feature_list("data\\dop\\drop_feature.txt")
    don_t_touch = _read_feature_list("data\\dop\\don_t_touch.txt")
    top30_feature = _read_feature_list("data\\dop\\top30_feature.txt")
    top100_feature = _read_feature_list("data\\dop\\top100_feature.txt")

    train.drop(columns=drop_features, inplace=True, errors="ignore")

    num_cols = [col for col in train.columns if col.startswith("num")]
    cat_cols = [col for col in train.columns if col.startswith("cat")]

    for col in num_cols:
        train[col] = train[col].astype("float32")

    for col in cat_cols:
        train[col] = train[col].astype("category")

    clip_cols = [col for col in num_cols if col not in don_t_touch]

    clip_bounds = {}
    for col in clip_cols:
        q_low = train[col].quantile(0.0005)
        q_high = train[col].quantile(0.9995)
        train[col] = train[col].clip(q_low, q_high)
        clip_bounds[col] = (q_low, q_high)

    medians = {}
    for col in num_cols:
        med = train[col].median()
        train[col] = train[col].fillna(med)
        medians[col] = med

    # ============
    # feature engineering on train
    # ============
    train = _add_pairwise_features(train, top30_feature)
    train, freq_maps, quantile_bins = _add_top100_base_features_train(train, top100_feature)
    train = _add_rowwise_aggregations(train, top100_feature)

    agg_maps = _build_category_agg_maps(train, cat_cols, top30_feature, max_cat_cols=5)
    train = _apply_category_aggs(train, agg_maps)

    train.replace([np.inf, -np.inf], np.nan, inplace=True)

    feature_tokens = [
        "_x_", "_sub_", "_div_", "_gt_",
        "_freq", "_isna", "_abs", "_rank_pct", "_qbin",
        "top100_", "_by_", "_minus_"
    ]

    final_num_cols_train = [
        col for col in train.columns
        if col != "customer_id" and (
            col.startswith("num") or any(token in col for token in feature_tokens)
        )
    ]

    feature_medians = {}
    for col in final_num_cols_train:
        if pd.api.types.is_numeric_dtype(train[col]):
            med = train[col].median()
            train[col] = train[col].fillna(med)
            feature_medians[col] = med

    _safe_float32(train, train.columns)

    if path_save_train:
        train.to_parquet(path_save_train, index=False)

    # ============
    # 2. TEST
    # ============
    df_m_test = pd.read_parquet(path_m_test)
    df_e_test = pd.read_parquet(path_e_test)
    test = df_m_test.merge(df_e_test, on="customer_id", how="left")
    del df_m_test, df_e_test

    test.drop(columns=drop_features, inplace=True, errors="ignore")

    generated_tokens = [
        "_x_", "_sub_", "_div_", "_gt_",
        "_freq", "_isna", "_abs", "_rank_pct", "_qbin",
        "top100_", "_by_", "_minus_"
    ]

    base_train_cols = [
        col for col in train.columns
        if not any(token in col for token in generated_tokens)
    ]

    missing_in_test = [col for col in base_train_cols if col not in test.columns]
    for col in missing_in_test:
        test[col] = pd.NA

    extra_in_test = [col for col in test.columns if col not in base_train_cols]
    if extra_in_test:
        test.drop(columns=extra_in_test, inplace=True, errors="ignore")

    test = test[base_train_cols]

    for col in num_cols:
        if col in test.columns:
            test[col] = test[col].astype("float32")

    for col in cat_cols:
        if col in test.columns:
            test[col] = test[col].astype("category")

    for col in clip_cols:
        if col in test.columns:
            q_low, q_high = clip_bounds[col]
            test[col] = test[col].clip(q_low, q_high)

    for col in num_cols:
        if col in test.columns:
            test[col] = test[col].fillna(medians[col])

    # ============
    # feature engineering on test
    # ============
    test = _add_pairwise_features(test, top30_feature)
    test = _add_top100_base_features_test(test, top100_feature, freq_maps, quantile_bins)
    test = _add_rowwise_aggregations(test, top100_feature)
    test = _apply_category_aggs(test, agg_maps)

    test.replace([np.inf, -np.inf], np.nan, inplace=True)

    missing_in_test = [col for col in train.columns if col not in test.columns]
    for col in missing_in_test:
        if col in train.columns and str(train[col].dtype) == "category":
            test[col] = pd.Series(pd.Categorical([pd.NA] * len(test)), index=test.index)
        else:
            test[col] = pd.Series(np.nan, index=test.index)

    extra_in_test = [col for col in test.columns if col not in train.columns]
    if extra_in_test:
        test.drop(columns=extra_in_test, inplace=True, errors="ignore")

    test = test[train.columns]

    for col, med in feature_medians.items():
        if col in test.columns:
            test[col] = test[col].fillna(med)

    _safe_float32(test, test.columns)

    if path_save_test:
        test.to_parquet(path_save_test, index=False)

    final_num_cols = [col for col in train.columns if pd.api.types.is_numeric_dtype(train[col])]
    final_cat_cols = [col for col in train.columns if str(train[col].dtype) == "category"]

    return train, test, final_num_cols, final_cat_cols

In [ ]:
# train, test, num_cols, cat_cols = processing_train_test(path_m_train, path_e_train, path_m_test, path_e_test, path_save_train, path_save_test)

In [ ]:
train = pd.read_parquet(path_save_train)
num_cols = list(train.columns[train])
cat_cols = list(train.columns[train.columns.str.startswith("cat")])

In [14]:
train_t = pd.read_parquet(f"{path_train}train_target.parquet")
target_cols = [col for col in train_t.columns if col != "customer_id"]

### Пробники

#### Загрузка данных

In [5]:
train_m = pd.read_parquet(f"{path_train}train_main_features.parquet")
train_e = pd.read_parquet(f"{path_train}train_extra_features.parquet")
train_t = pd.read_parquet(f"{path_train}train_target.parquet")

In [6]:
train = train_m.merge(train_e, on="customer_id", how="left")


In [ ]:
# test_m = pd.read_parquet(f"{path_test}test_main_features.parquet")
# test_e = pd.read_parquet(f"{path_test}test_extra_features.parquet")

#### Столбцы для удаления

In [7]:
drop_features1 = list(train.isnull().sum().sort_values(ascending=False)[:600].index)

In [8]:
drop_features2 = ['num_feature_559',
 'num_feature_563',
 'num_feature_568',
 'num_feature_573',
 'num_feature_1809',
 'num_feature_541',
 'num_feature_373',
 'num_feature_2065',
 'num_feature_973',
 'num_feature_992',
 'num_feature_980',
 'num_feature_556',
 'num_feature_1804',
 'num_feature_1799',
 'num_feature_2032',
 'num_feature_1989',
 'num_feature_350',
 'num_feature_416',
 'num_feature_346',
 'num_feature_336',
 'num_feature_1737',
 'num_feature_1006',
 'num_feature_2103',
 'num_feature_1739',
 'num_feature_2101',
 'num_feature_1727',
 'num_feature_2095',
 'num_feature_1746',
 'num_feature_388',
 'num_feature_378',
 'num_feature_1791',
 'num_feature_397',
 'num_feature_1035',
 'num_feature_1081',
 'num_feature_2113',
 'num_feature_2339',
 'num_feature_1461',
 'num_feature_1453',
 'num_feature_1465',
 'num_feature_1466',
 'num_feature_2311',
 'num_feature_2315',
 'num_feature_2329',
 'num_feature_1937',
 'num_feature_695',
 'num_feature_510',
 'num_feature_496',
 'num_feature_700',
 'num_feature_1767',
 'num_feature_1769',
 'num_feature_1773',
 'num_feature_2051',
 'num_feature_2056',
 'num_feature_2033',
 'num_feature_455',
 'num_feature_2049',
 'num_feature_519',
 'num_feature_1996',
 'num_feature_432',
 'num_feature_2367',
 'cat_feature_17',
 'cat_feature_28',
 'num_feature_1442',
 'num_feature_2360',
 'num_feature_308',
 'num_feature_2132',
 'num_feature_2133',
 'num_feature_1644',
 'num_feature_1657',
 'num_feature_1646',
 'num_feature_1138',
 'num_feature_1105',
 'num_feature_871',
 'num_feature_1108',
 'num_feature_865',
 'num_feature_869',
 'num_feature_847',
 'num_feature_872',
 'num_feature_1819',
 'num_feature_608',
 'num_feature_1050',
 'num_feature_286',
 'num_feature_1822',
 'num_feature_829',
 'num_feature_832',
 'num_feature_830',
 'num_feature_1122',
 'num_feature_281',
 'num_feature_1047',
 'num_feature_1689',
 'num_feature_354',
 'num_feature_305',
 'num_feature_2112',
 'num_feature_294',
 'num_feature_2163',
 'num_feature_2159',
 'num_feature_2188',
 'num_feature_2191',
 'num_feature_266',
 'num_feature_270',
 'num_feature_271',
 'num_feature_2172',
 'num_feature_2176',
 'num_feature_2186',
 'num_feature_243',
 'num_feature_246',
 'num_feature_238',
 'num_feature_2185',
 'num_feature_1236',
 'num_feature_1231',
 'num_feature_1239',
 'num_feature_1189',
 'num_feature_779',
 'num_feature_1840',
 'num_feature_658',
 'num_feature_1958',
 'num_feature_653',
 'num_feature_597',
 'num_feature_843',
 'num_feature_1612',
 'num_feature_1605',
 'num_feature_2155',
 'num_feature_2158',
 'num_feature_1618',
 'num_feature_1221',
 'num_feature_1863',
 'num_feature_782',
 'num_feature_1631',
 'num_feature_758',
 'num_feature_763',
 'num_feature_774',
 'num_feature_1893',
 'num_feature_2245',
 'num_feature_115',
 'num_feature_195',
 'num_feature_2192',
 'num_feature_1557',
 'num_feature_1561',
 'num_feature_1555',
 'num_feature_180',
 'num_feature_143',
 'num_feature_2184',
 'num_feature_2225',
 'num_feature_1573',
 'num_feature_1598',
 'num_feature_1546',
 'num_feature_1547',
 'num_feature_1363',
 'num_feature_1535',
 'num_feature_80',
 'num_feature_1543',
 'num_feature_2268',
 'num_feature_2233',
 'num_feature_628',
 'num_feature_1336',
 'num_feature_2243',
 'num_feature_618',
 'num_feature_1948',
 'num_feature_1946',
 'num_feature_1878',
 'num_feature_1426',
 'num_feature_106',
 'num_feature_1514',
 'num_feature_2261',
 'num_feature_1525',
 'num_feature_1534',
 'num_feature_1378',
 'num_feature_1372',
 'cat_feature_50',
 'cat_feature_54',
 'cat_feature_57',
 'num_feature_672',
 'num_feature_666',
 'num_feature_1899',
 'cat_feature_59',
 'num_feature_1480',
 'num_feature_2280',
 'num_feature_1410',
 'num_feature_89',
 'num_feature_2279',
 'cat_feature_29',
 'num_feature_2347',
 'num_feature_2355',
 'num_feature_32',
 'num_feature_716',
 'cat_feature_14',
 'num_feature_733',
 'num_feature_744',
 'num_feature_1901',
 'num_feature_2373',
 'num_feature_2335',
 'cat_feature_31']

In [9]:
drop_features = drop_features1 + drop_features2

#### Удаление слабых фич

In [9]:
train.drop(columns=drop_features, inplace=True)
train.drop(columns=["customer_id"], inplace=True)

In [10]:
num_cols = list(train.columns[train.columns.str.startswith("num")])
cat_cols = list(train.columns[train.columns.str.startswith("cat")])
target_cols = [col for col in train_t.columns if col != "customer_id"]

In [11]:
print(f"Количество числовых столбцов: {len(num_cols)}, количество категориальных столбцов: {len(cat_cols)}")

Количество числовых столбцов: 1590, количество категориальных столбцов: 58


#### Перевод типов

In [12]:
for num_col in num_cols:
    train[num_col] = train[num_col].astype("float32")

for cat_col in cat_cols:
    train[cat_col] = train[cat_col].astype("category")

#### Заполнение пропусков и разбиение данных

In [13]:
for column in num_cols:
    train[column] = train[column].fillna(train[column].median())

In [ ]:
train_x, val_x, train_y, val_y = train_test_split(train, train_t, test_size=0.2, random_state=42)

In [51]:
top29_features = ['cat_feature_39',
 'cat_feature_52',
 'num_feature_41',
 'num_feature_117',
 'num_feature_76',
 'num_feature_27',
 'num_feature_1106',
 'num_feature_62',
 'num_feature_1234',
 'num_feature_85',
 'num_feature_29',
 'num_feature_879',
 'num_feature_24',
 'num_feature_58',
 'num_feature_35',
 'num_feature_126',
 'num_feature_15',
 'num_feature_2343',
 'num_feature_147',
 'num_feature_1633',
 'num_feature_1787',
 'num_feature_525',
 'num_feature_961',
 'num_feature_2353',
 'num_feature_1459',
 'num_feature_1334',
 'num_feature_87',
 'num_feature_327',
 'num_feature_1392']

# Основные функции

In [6]:
def validation(model, val_y):
    predict = model.predict_proba(val_x)[:,1]

    score = roc_auc_score(val_y, predict)

    print(f"{score:.6f}\n")

    return score

In [7]:
def loading_unloading_wrapper(path_model, code_study_f, params, train_y, val_y, cat_cols = cat_cols):
    print(f"{val_y.name}")
    if not os.path.exists(path_model):
        model = code_study_f(train_y, val_y, cat_cols, **params)
        joblib.dump(model, path_model)
    else:
        model = joblib.load(path_model)

    score = validation(model, val_y)

    return score

In [8]:
def study_model_all_feature(train_y, val_y, cat_cols, **params):
    model = lgb.LGBMClassifier(**params)

    model.fit(train_x,
            train_y,
            eval_set=[(val_x, val_y)],
            eval_metric="auc",
            categorical_feature=cat_cols,
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=100),
            ],
            )

    return model

In [9]:
def feature_importance(model):
    summ_imp = sum(model.feature_importances_)
    feat_imp_df = pd.DataFrame({
        "feature": model.feature_name_,
        "importance": model.feature_importances_ / summ_imp
    })

    return feat_imp_df.sort_values("importance", ascending=False)

In [10]:
def all_feature_importance(paths):
    model = joblib.load(paths[0])
    feature_importance_table = pd.DataFrame({
        "feature": model.feature_name_,
        "importance": 0.0
    }).set_index("feature")

    for path in paths:
        model = joblib.load(path)
        feature_imp = feature_importance(model)
        for row in feature_imp.itertuples():
            feature_importance_table.loc[row.feature, "importance"] += row.importance

    feature_importance_table["importance"] /= len(paths)

    return feature_importance_table.reset_index().sort_values("importance", ascending=False)

# Параметры

In [21]:
params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",

    "learning_rate": 0.06,
    "n_estimators": 6000,

    "num_leaves": 95,
    "max_depth": 9,

    "min_child_samples": 180,
    "min_split_gain": 0.01,

    "subsample": 0.75,
    "subsample_freq": 1,
    "colsample_bytree": 0.35,

    "reg_alpha": 2.0,
    "reg_lambda": 12.0,

    "max_bin": 127,

    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

# Обучение

In [16]:
train_x, val_x, train_y, val_y = train_test_split(train, train_t, test_size=0.2, random_state=42)

In [17]:
del train

In [22]:
scores = []
for target in target_cols:
    train_y_ = train_y[target]
    val_y_ = val_y[target]
    score = loading_unloading_wrapper(Path(f"{path_save_model_with_extra_feature2}{target}.pkl"), study_model_all_feature, params, train_y_, val_y_, cat_cols)
    scores.append(score)

print(f"Macro_average: {sum(scores) / len(scores)}")

target_1_1
[100]	valid_0's auc: 0.907828
[200]	valid_0's auc: 0.910282
[300]	valid_0's auc: 0.910017
0.910663

target_1_2
[100]	valid_0's auc: 0.826466
0.827695

target_1_3
[100]	valid_0's auc: 0.873361
[200]	valid_0's auc: 0.875718
[300]	valid_0's auc: 0.875696
0.875917

target_1_4
[100]	valid_0's auc: 0.833146
[200]	valid_0's auc: 0.836832
[300]	valid_0's auc: 0.836912
0.837311

target_1_5
[100]	valid_0's auc: 0.885937
0.886061

target_2_1
[100]	valid_0's auc: 0.82734
[200]	valid_0's auc: 0.827215
0.829256

target_2_2
[100]	valid_0's auc: 0.937359
[200]	valid_0's auc: 0.939022
[300]	valid_0's auc: 0.939093
[400]	valid_0's auc: 0.939334
[500]	valid_0's auc: 0.939055
0.939379

target_2_3
[100]	valid_0's auc: 0.788443
0.796625

target_2_4
[100]	valid_0's auc: 0.755901
[200]	valid_0's auc: 0.755247
0.758797

target_2_5
[100]	valid_0's auc: 0.74302
[200]	valid_0's auc: 0.734494
0.743992

target_2_6
[100]	valid_0's auc: 0.761183
0.761702

target_2_7
[100]	valid_0's auc: 0.854918
[200]	vali